In [ ]:
# Install if needed
!pip install datasets

# Imports
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from datasets import load_dataset
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
print("Loading Amazon review dataset...")

train_data = load_dataset("amazon_polarity", split="train[:15000]")
test_data = load_dataset("amazon_polarity", split="test[:4000]")

train_texts = train_data['content']
train_labels = train_data['label']

test_texts = test_data['content']
test_labels = test_data['label']

Loading Amazon review dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

amazon_polarity/train-00000-of-00004.par(…):   0%|          | 0.00/260M [00:00<?, ?B/s]

amazon_polarity/train-00001-of-00004.par(…):   0%|          | 0.00/258M [00:00<?, ?B/s]

amazon_polarity/train-00002-of-00004.par(…):   0%|          | 0.00/255M [00:00<?, ?B/s]

amazon_polarity/train-00003-of-00004.par(…):   0%|          | 0.00/254M [00:00<?, ?B/s]

amazon_polarity/test-00000-of-00001.parq(…):   0%|          | 0.00/117M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3600000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/400000 [00:00<?, ? examples/s]

In [ ]:
VOCAB_LIMIT = 12000
MAX_SEQUENCE = 120

print("Processing text...")

tokenizer = Tokenizer(num_words=VOCAB_LIMIT, oov_token="<UNK>")
tokenizer.fit_on_texts(train_texts)

train_sequences = tokenizer.texts_to_sequences(train_texts)
test_sequences = tokenizer.texts_to_sequences(test_texts)

train_padded = pad_sequences(train_sequences, maxlen=MAX_SEQUENCE, padding='post')
test_padded = pad_sequences(test_sequences, maxlen=MAX_SEQUENCE, padding='post')

Processing text...


In [ ]:
X_train_t = torch.LongTensor(train_padded)
y_train_t = torch.FloatTensor(train_labels)

X_test_t = torch.LongTensor(test_padded)
y_test_t = torch.FloatTensor(test_labels)

BATCH_SIZE = 128

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=BATCH_SIZE)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

Device: cpu


In [ ]:
class ReviewGRU(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim)

        self.gru_layer = nn.GRU(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )

        self.classifier = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        x = self.embedding(x)

        _, hidden = self.gru_layer(x)

        hidden = hidden[-1]  # last layer hidden state

        out = self.classifier(hidden)
        return out.squeeze()

In [ ]:
model = ReviewGRU(VOCAB_LIMIT).to(device)

loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 6

In [ ]:
print("Training started...\n")

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    correct = 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = model(inputs)
        loss = loss_fn(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        preds = torch.sigmoid(outputs)
        preds = (preds >= 0.5).float()
        correct += (preds == labels).sum().item()

    avg_loss = total_loss / len(train_loader)
    accuracy = correct / len(train_loader.dataset)

    print(f"Epoch {epoch+1} | Loss: {avg_loss:.4f} | Accuracy: {accuracy*100:.2f}%")

Training started...

Epoch 1 | Loss: 0.6937 | Accuracy: 51.76%
Epoch 2 | Loss: 0.5907 | Accuracy: 68.97%
Epoch 3 | Loss: 0.4201 | Accuracy: 81.58%
Epoch 4 | Loss: 0.3109 | Accuracy: 87.46%
Epoch 5 | Loss: 0.2300 | Accuracy: 91.55%
Epoch 6 | Loss: 0.1735 | Accuracy: 94.09%


In [ ]:
model.eval()

test_loss = 0
correct = 0

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        outputs = model(inputs)
        loss = loss_fn(outputs, labels)

        test_loss += loss.item()

        preds = torch.sigmoid(outputs)
        preds = (preds >= 0.5).float()
        correct += (preds == labels).sum().item()

final_loss = test_loss / len(test_loader)
final_acc = correct / len(test_loader.dataset)

print("\nFinal Results:")
print(f"Test Loss: {final_loss:.4f}")
print(f"Test Accuracy: {final_acc*100:.2f}%")


Final Results:
Test Loss: 0.4553
Test Accuracy: 83.75%
